# Fig. 1j: PC attractors in the reduced coordinates $(x_0, w_{\rm diff}, x_{\rm diff})$

Left/centre of Fig. 1j: deterministic flows of the reduced system for $s=1$ and $s=0$ ($\alpha=\beta=0.01$). Right: the full stochastic model on the deterministic bandit ($\alpha=0.02$, $\beta=0.03$, $\eta=0.008$, activities capped at 3; 2,000-step burn-in, 40,000 steps, 3 seeds).

PC = prospective configuration. Reduced equations (the companion paper (Li & Erlich)):

$$\dot x_0=2\alpha(s-x_0),\quad \dot x_{\rm diff}=2\alpha(x_0 w_{\rm diff}-x_{\rm diff}),\quad \dot w_{\rm diff}=\beta x_0(x_{\rm diff}-x_0 w_{\rm diff}).$$

In [ ]:
import numpy as np
import matplotlib as mpl, matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from mpl_toolkits.mplot3d.art3d import Line3DCollection
np.set_printoptions(precision=4, suppress=True)

alpha = beta = eta = 0.01

# d/dt of state v = (x0, xdiff, wdiff)
def F_PC(v, s):
    x0, xd, wd = v
    return np.array([2*alpha*(s-x0),
                     2*alpha*(x0*wd - xd),
                     beta*x0*(xd - x0*wd)])

print("alpha = beta = eta =", alpha)
print("PC:  dx0=2a(s-x0)   dxdiff=2a(x0*wdiff-xdiff)   dwdiff=w*x0*(xdiff - x0*wdiff)")


## Fixed points and linear stability

The line $\{x_0=s,\ x_{\rm diff}=s\,w_{\rm diff}\}$ is a set of fixed points: $\dot x_0=\dot x_{\rm diff}=0$ there, and $\dot w_{\rm diff}=0$ because the output error vanishes. The cell below prints the Jacobian eigenvalues at a point on the line; one eigenvalue is 0 (marginal stability along $w_{\rm diff}$).

In [ ]:
def jac(F, v, s, h=1e-6):
    J = np.zeros((3,3))
    for i in range(3):
        dp, dm = v.copy(), v.copy(); dp[i]+=h; dm[i]-=h
        J[:,i] = (F(dp,s)-F(dm,s))/(2*h)
    return J

print(f"{'rule':4} {'s':>3} {'F at fixed pt':>22}   eigenvalues")
for name, F in [('PC',F_PC)]:
    for s in (0.0, 1.0):
        c = 0.4                                   # any point on the candidate line
        v = np.array([s, s*c, c])                 # x0=s, xdiff=s*wdiff
        ev = np.sort(np.linalg.eigvals(jac(F, v, s)).real)
        print(f"{name:4} {s:>3} {str(np.round(F(v,s),6)):>22}   {np.round(ev,4)}")

## 4. Phase portraits — axes $(x_0,\,w_{\rm diff},\,x_{\rm diff})$

Streams colored by speed (`plasma`); the **red line** is the fixed-point set.

In [ ]:
Lx, Lw, Lz = 1.5, 1.0, 1.0
import matplotlib.colors as mcolors
# plasma but truncated before bright yellow, fading into a light orange tail
# full warm gradient: purple -> red -> orange -> light yellow (top).
# Paired with a power-law speed norm so the orange transition is visible.
_pl = mpl.cm.plasma
cmap = mcolors.LinearSegmentedColormap.from_list(
    'warm_grad', [_pl(0.05), _pl(0.35), _pl(0.55), '#f5944a', '#f9cb6e'], N=256)

def style3d(ax, title):
    ax.set_box_aspect((1,1,1))
    for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
        axis.set_pane_color((1,1,1,0)); axis.pane.set_edgecolor((0,0,0,0))
        axis._axinfo['grid']['linewidth'] = 0.0
        axis.set_major_locator(mpl.ticker.MaxNLocator(3))
    ax.tick_params(labelsize=7, width=0.25, length=2, pad=-3)
    ax.set_xlim(0,Lx); ax.set_ylim(-Lw,Lw); ax.set_zlim(-Lz,Lz)
    ax.set_xlabel(r'$x_0$', fontsize=9, labelpad=-7)
    ax.set_ylabel(r'$w_{\rm diff}$', fontsize=9, labelpad=-6)
    ax.set_zlabel(r'$x_{\rm diff}$', fontsize=9, labelpad=-6)
    # light wireframe box as a visual anchor
    import itertools
    xs, ys, zs = (0, Lx), (-Lw, Lw), (-Lz, Lz)
    corners = list(itertools.product(xs, ys, zs))
    for p, q in itertools.combinations(corners, 2):
        if sum(a != b for a, b in zip(p, q)) == 1:        # cube edges only
            ax.plot(*zip(p, q), color='#bcbcbc', lw=0.4, zorder=0)
    ax.view_init(elev=18, azim=-45); ax.set_title(title, fontsize=9, pad=-2)

def panel(ax, F, s, title, seed=0, n_traj=46, T=600.0):
    style3d(ax, title)
    wline = np.array([-Lw, Lw])                      # red fixed-point line
    ax.plot([s,s], wline, s*wline, color='#e8431f', lw=3.2,
            solid_capstyle='round', zorder=6)
    # regular grid of seed points (state order x0, xdiff, wdiff)
    gx0 = np.linspace(0.30, Lx-0.10, 3)
    gxd = np.linspace(-Lz*0.85, Lz*0.85, 4)
    gwd = np.linspace(-Lw*0.85, Lw*0.85, 4)
    seeds = [np.array([a,b,c]) for a in gx0 for b in gxd for c in gwd]
    paths, speeds = [], []
    for v0 in seeds:
        sol = solve_ivp(lambda t,v: F(v,s), [0,T], v0,
                        t_eval=np.linspace(0,T,70), rtol=1e-7, atol=1e-9)
        P = sol.y.T                                  # x0, xdiff, wdiff
        Q = np.column_stack([P[:,0], P[:,2], P[:,1]])# -> x0, wdiff, xdiff
        m = (Q[:,0]>=-0.02)&(np.abs(Q[:,1])<=Lw*1.02)&(np.abs(Q[:,2])<=Lz*1.02)
        Q = Q[m]
        if len(Q) < 4: continue
        spd = np.linalg.norm(np.diff(Q,axis=0), axis=1)
        paths.append((Q,spd)); speeds.append(spd)
    norm = mpl.colors.PowerNorm(1.7, 0, np.percentile(np.concatenate(speeds),92)+1e-9)
    for Q,spd in paths:
        ax.add_collection3d(Line3DCollection(np.stack([Q[:-1],Q[1:]],1),
                            colors=cmap(norm(spd)), linewidths=1.2))
        for i in np.linspace(2, len(Q)-2, 3).astype(int):
            d = Q[i+1]-Q[i]; nd = np.linalg.norm(d)
            if nd < 1e-9: continue
            d = d/nd*0.13
            ax.quiver(*Q[i], *d, color=cmap(norm(spd[min(i,len(spd)-1)])),
                      arrow_length_ratio=0.6, linewidth=1.1)

import os; os.makedirs('svgs', exist_ok=True)
fig = plt.figure(figsize=(6.6, 3.0))
panel(fig.add_subplot(1,2,1,projection='3d'), F_PC, 1.0, 'PC   s=1 only')
panel(fig.add_subplot(1,2,2,projection='3d'), F_PC, 0.0, 'PC   s=0 only')
plt.tight_layout(w_pad=0.2)
plt.savefig('svgs/f6c_phase_portraits.svg', bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np, itertools, jax, jax.numpy as jnp
import matplotlib.pyplot as plt
from ssb.models import ssb, rules
from ssb.utils.batch_inits import make_default_hps, batchify_hps, init_params_batch
from ssb.envs.bandits import bandit_det

# ---- config ----
PALETTE = ['#7991b5', '#f4bc67', '#d76577', '#74ad83', '#967fb7']
SEEDS   = [8, 24, 23, 6, 10]                   # example seeds
ALPHA, BETA, ETA = 0.02, 0.03, 0.008           # same for both rules
BURN, SAMPLE, N_SHOW = 2000, 40000, 2200
Lxc, Lwc, Lzc = 1.5, 0.9, 0.9
XTc, YTc, ZTc = [0, .5, 1, 1.5], [-.8, 0, .8], [-.8, 0, .8]
AZ = -48

def sim_cloud(rule, seed):
    """Steady-state cloud of the full stochastic model on the deterministic bandit.
    Returns time series of (x0, w_diff = w2-w1, x_diff = x2-x1)."""
    hps = make_default_hps(seed)
    hps['alpha']=jnp.array([ALPHA]); hps['beta']=jnp.array([BETA]); hps['eta_a']=jnp.array([ETA])
    hb = batchify_hps(hps, 1)
    uw, us = rules.rule2fns(rule)
    UA=jax.vmap(ssb.update_acts); UW=jax.vmap(uw); US=jax.vmap(us)
    NO=jax.vmap(ssb.act_noise);   EX=jax.vmap(ssb.expand_stimulus)
    env=jax.vmap(bandit_det, in_axes=(0,0,0,0))
    acts,states,weights,tern,key = init_params_batch(hb)
    rewards_b=jnp.tile(jnp.array([0.,1.0])[None],(1,1)); dummy=jnp.zeros((1,1))
    @jax.jit
    def step(c,_):
        acts,states,weights,key=c
        rb,action,_,key=env(dummy,acts[:,-2:],rewards_b,key); stim=EX(rb,acts)
        acts=UA(stim,acts,states,weights,tern,hb); acts,key=NO(acts,key,hb); acts=jnp.clip(acts,0.,3.)
        weights=UW(stim,acts,states,weights,tern,hb); states=US(acts,states,hb)
        return (acts,states,weights,key),(acts[:,0], weights[:,0,2]-weights[:,0,1], acts[:,2]-acts[:,1])
    carry,_ = jax.lax.scan(step,(acts,states,weights,key),None,length=BURN)
    _,out   = jax.lax.scan(step,carry,None,length=SAMPLE)
    return [np.asarray(o).ravel() for o in out]

def style3d_cloud(ax, title):
    ax.set_box_aspect((1,1,1))
    for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
        axis.set_pane_color((1,1,1,0)); axis.pane.set_edgecolor((0,0,0,0))
        axis._axinfo['grid']['linewidth'] = 0.0
    ax.set_xlim(0,Lxc); ax.set_ylim(-Lwc,Lwc); ax.set_zlim(-Lzc,Lzc)
    ax.set_xticks(XTc); ax.set_yticks(YTc); ax.set_zticks(ZTc)
    ax.tick_params(labelsize=7, width=0.25, length=2, pad=-3)
    ax.set_xlabel(r'$x_0$', fontsize=9, labelpad=-7)
    ax.set_ylabel(r'$w_{\rm diff}$', fontsize=9, labelpad=-6)
    ax.set_zlabel(r'$x_{\rm diff}$', fontsize=9, labelpad=-6)
    for p,q in itertools.combinations(list(itertools.product((0,Lxc),(-Lwc,Lwc),(-Lzc,Lzc))),2):
        if sum(a!=b for a,b in zip(p,q))==1: ax.plot(*zip(p,q),color='#bcbcbc',lw=0.4,zorder=0)
    ax.view_init(elev=18, azim=AZ); ax.set_title(title, fontsize=9, pad=-2)


In [ ]:
# Fig. 1j right: PC, full environment, 3 seeds (sim_cloud / style3d_cloud defined above)
from mpl_toolkits.mplot3d.art3d import Line3DCollection

SEEDS3 = [8, 20, 11]
COLORS3 = ['#7991b5', '#f4bc67', '#d76577']        # notebook PALETTE[:3]

# cache the three time series per rule so both renderings use identical runs
_TS = {f'PC_{s}': sim_cloud('PC', s) for s in SEEDS3}

def _redline(ax):
    wl = np.array([-Lwc, Lwc])
    ax.plot([1, 1], wl, wl, color='#e8431f', lw=2.2, solid_capstyle='round', zorder=2)

def _legend(ax):
    for c, s in zip(COLORS3, SEEDS3):
        ax.plot([], [], color=c, lw=1.5, label=f'seed {s}')
    ax.legend(fontsize=7, frameon=False, loc='upper left',
              bbox_to_anchor=(0.82, 0.96), handlelength=1.0, borderaxespad=0)

def draw_cloud3(ax, rule):
    style3d_cloud(ax, rule); _redline(ax)
    rng = np.random.default_rng(0)
    for c, s in zip(COLORS3, SEEDS3):
        x, W, X = _TS[f'{rule}_{s}']
        idx = rng.choice(len(x), min(2200, len(x)), replace=False)
        ax.scatter(x[idx], W[idx], X[idx], s=1.6, c=c, alpha=0.30,
                   linewidths=0, rasterized=True, zorder=4)
    _legend(ax)

fig = plt.figure(figsize=(3.3, 3.2))
draw_cloud3(fig.add_subplot(1, 1, 1, projection='3d'), 'PC')
plt.tight_layout()
plt.savefig('svgs/f6c_full_env_3seeds.svg', dpi=300, bbox_inches='tight')
plt.show()